# Weather MLOps — Exploration `weather_final.csv`

Notebook d'exploration classique du dataset final exporté par le pipeline.  
Source : `data/output/weather_final.csv` — 174 018 lignes × 46 colonnes.

**Sections :**
1. Chargement & aperçu général
2. Qualité des données (nulls, couverture temporelle, villes)
3. Distributions des variables météo brutes
4. Analyse des cibles (targets ML)
5. Saisonnalité & patterns temporels
6. Géographie — patterns par ville et état
7. Corrélations entre features
8. Analyse des prédictions du modèle
9. Synthèse

**Prérequis :**
```bash
pip install pandas matplotlib seaborn scipy
```

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='tab10')

CSV_PATH = Path('..') / 'data' / 'output' / 'weather_final.csv'
assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH}"

df = pd.read_csv(CSV_PATH, parse_dates=['date'])
df = df.sort_values(['city', 'date']).reset_index(drop=True)

print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période        : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Villes         : {df['city'].nunique()} villes / {df['state'].nunique()} états")

## 1. Aperçu général

In [ ]:
# Catalogue des colonnes par groupe fonctionnel
COLS_GEO    = ['date', 'city', 'state', 'latitude', 'longitude']
COLS_TEMP   = ['min_temp', 'max_temp', 'temp_9am', 'temp_3pm']
COLS_RAIN   = ['rainfall', 'rain_sum', 'precipitation_hours', 'rain_today']
COLS_WIND   = ['wind_gust_speed', 'wind_speed_9am', 'wind_speed_3pm', 'wind_speed_100m_9am', 'wind_speed_100m_3pm']
COLS_HUMID  = ['humidity_9am', 'humidity_3pm', 'dew_point_9am', 'dew_point_3pm', 'evaporation']
COLS_PRESS  = ['pressure_9am', 'pressure_3pm', 'surface_pressure_9am', 'surface_pressure_3pm']
COLS_CLOUD  = ['cloud_9am', 'cloud_3pm', 'sunshine_hours', 'shortwave_radiation_sum']
COLS_VPD    = ['vpd_9am', 'vpd_3pm']
COLS_TARGET = ['rain_tomorrow', 'rain_tomorrow_proba', 'max_temp_tomorrow',
                'weather_type_tomorrow', 'comfort_score', 'heatwave_risk', 'frost_risk', 'storm_probability']

print("Groupes de colonnes :")
for name, cols in [
    ('Géographie / index', COLS_GEO),
    ('Température', COLS_TEMP),
    ('Précipitations', COLS_RAIN),
    ('Vent', COLS_WIND),
    ('Humidité / évaporation', COLS_HUMID),
    ('Pression', COLS_PRESS),
    ('Nuages / rayonnement', COLS_CLOUD),
    ('VPD', COLS_VPD),
    ('Cibles modèle', COLS_TARGET),
]:
    print(f"  {name:<26}: {cols}")

In [ ]:
df.describe().T

## 2. Qualité des données

In [ ]:
# Taux de nulls par colonne
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['tomato' if v > 50 else ('orange' if v > 10 else 'steelblue') for v in null_pct.values]
null_pct.plot(kind='bar', ax=ax, color=colors)
ax.axhline(50, color='red', linestyle='--', linewidth=1, label='50%')
ax.axhline(10, color='orange', linestyle=':', linewidth=1, label='10%')
ax.set_title('Taux de valeurs manquantes par colonne')
ax.set_ylabel('% NaN')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nRaison des NULLs massifs (~80% des lignes) :")
print("  Les colonnes Open-Meteo enrichies (rain_sum, dew_point, surface_pressure, shortwave,")
print("  vpd, wind_100m, precipitation_hours) ne couvrent que les données récentes (~33 550 lignes).")
print(f"  Ratio : {33550/174018:.1%} du dataset — lignes avec ces colonnes renseignées.")

In [ ]:
# Couverture temporelle par ville
city_coverage = (
    df.groupby('city')['date']
    .agg(['min', 'max', 'count'])
    .rename(columns={'min': 'debut', 'max': 'fin', 'count': 'n_jours'})
    .sort_values('n_jours', ascending=False)
)
city_coverage['n_jours_attendus'] = (city_coverage['fin'] - city_coverage['debut']).dt.days + 1
city_coverage['completude_pct'] = city_coverage['n_jours'] / city_coverage['n_jours_attendus'] * 100

print(f"Villes : {len(city_coverage)}")
print(f"Complétude min : {city_coverage['completude_pct'].min():.1f}%")
print(f"Complétude moy : {city_coverage['completude_pct'].mean():.1f}%")
print()
print(city_coverage.to_string())

In [ ]:
# Heatmap présence de données par ville × année
df['year'] = df['date'].dt.year
hm_data = df.groupby(['city', 'year']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(hm_data, ax=ax, cmap='YlGn', linewidths=0.2,
            cbar_kws={'label': 'Nombre de jours'})
ax.set_title('Couverture de données : jours disponibles par ville × année')
ax.set_xlabel('Année')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 3. Distributions des variables météo brutes

In [ ]:
# Températures
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
temp_cols = COLS_TEMP
for ax, col in zip(axes, temp_cols):
    ax.hist(df[col].dropna(), bins=60, color='tomato', alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Moy {df[col].mean():.1f}°C')
    ax.set_title(col.replace('_', ' '))
    ax.set_xlabel('°C')
    ax.legend(fontsize=8)
plt.suptitle('Distributions des températures', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Précipitations — distribution log (forte asymétrie droite)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rain = df['rainfall']
axes[0].hist(rain[rain > 0], bins=80, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Précipitations > 0 (échelle linéaire)')
axes[0].set_xlabel('mm')

axes[1].hist(np.log1p(rain[rain > 0]), bins=80, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_title('Précipitations > 0 (log1p scale)')
axes[1].set_xlabel('log1p(mm)')

plt.suptitle(f'Rainfall — {(rain == 0).mean():.1%} de jours sans pluie, max = {rain.max():.0f} mm', fontweight='bold')
plt.tight_layout()
plt.show()

print("Percentiles précipitations (jours pluvieux) :")
print(rain[rain > 0].quantile([0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_string())

In [ ]:
# Humidité, vent, pression
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

pairs = [
    ('humidity_9am', 'humidity_3pm', 'steelblue', 'Humidité (%)'),
    ('wind_speed_9am', 'wind_speed_3pm', 'mediumseagreen', 'Vent surface (km/h)'),
    ('pressure_9am', 'pressure_3pm', 'mediumpurple', 'Pression MSLP (hPa)'),
]

for i, (col_9, col_3, color, title) in enumerate(pairs):
    axes[0, i].hist(df[col_9].dropna(), bins=50, alpha=0.7, color=color, label='9am', edgecolor='white')
    axes[0, i].hist(df[col_3].dropna(), bins=50, alpha=0.5, color='gray', label='3pm', edgecolor='white')
    axes[0, i].set_title(f'{title} — 9am vs 3pm')
    axes[0, i].legend()

# Vent gust
axes[1, 0].hist(df['wind_gust_speed'].dropna(), bins=60, color='tomato', alpha=0.8, edgecolor='white')
axes[1, 0].set_title('Vitesse des rafales (km/h)')

# Ensoleillement
axes[1, 1].hist(df['sunshine_hours'].dropna(), bins=50, color='gold', alpha=0.9, edgecolor='white')
axes[1, 1].set_title('Heures d\'ensoleillement')

# Nuages
axes[1, 2].hist(df['cloud_9am'].dropna(), bins=30, alpha=0.7, color='steelblue', label='9am', edgecolor='white')
axes[1, 2].hist(df['cloud_3pm'].dropna(), bins=30, alpha=0.5, color='gray', label='3pm', edgecolor='white')
axes[1, 2].set_title('Couverture nuageuse (oktas)')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Directions de vent (rose des vents simplifiée)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['wind_gust_dir', 'wind_dir_9am', 'wind_dir_3pm']):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue', alpha=0.8)
    ax.set_title(col.replace('_', ' '))
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Fréquence des directions de vent', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Colonnes Open-Meteo enrichies (données récentes uniquement)
df_recent = df[df['rain_sum'].notna()].copy()
print(f"Données enrichies (Open-Meteo) : {len(df_recent):,} lignes")
print(f"Période : {df_recent['date'].min().date()} → {df_recent['date'].max().date()}")

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

cols_recent = [
    ('rain_sum', 'steelblue', 'Rain sum (mm)'),
    ('precipitation_hours', 'mediumslateblue', 'Heures de précipitation'),
    ('shortwave_radiation_sum', 'gold', 'Rayonnement solaire (Wh/m²)'),
    ('vpd_9am', 'tomato', 'VPD 9am (kPa)'),
    ('vpd_3pm', 'coral', 'VPD 3pm (kPa)'),
    ('wind_speed_100m_9am', 'mediumseagreen', 'Vent 100m 9am (km/h)'),
]

for ax, (col, color, label) in zip(axes.flat, cols_recent):
    data = df_recent[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Moy {data.mean():.2f}')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle('Distributions des variables Open-Meteo (données récentes)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Analyse des cibles (targets ML)

In [ ]:
# Équilibre des classes pour les cibles binaires / catégorielle
df_target = df.dropna(subset=['rain_tomorrow'])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# rain_tomorrow
rc = df_target['rain_tomorrow'].value_counts().sort_index()
axes[0].bar(['Pas pluie (0)', 'Pluie (1)'], rc.values, color=['steelblue', 'tomato'])
axes[0].set_title(f'rain_tomorrow\nImbalance: {rc[1]/len(df_target):.1%} positifs')
for bar, val in zip(axes[0].patches, rc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontsize=9)

# heatwave_risk
hw = df_target['heatwave_risk'].round().value_counts().sort_index()
axes[1].bar(['Non (0)', 'Oui (1)'], hw.values, color=['steelblue', 'tomato'])
axes[1].set_title(f'heatwave_risk\n{hw.get(1,0)/len(df_target):.1%} positifs')

# frost_risk
fr = df_target['frost_risk'].round().value_counts().sort_index()
axes[2].bar(['Non (0)', 'Oui (1)'], fr.values, color=['steelblue', 'lightskyblue'])
axes[2].set_title(f'frost_risk\n{fr.get(1,0)/len(df_target):.1%} positifs')

# weather_type_tomorrow
wt = df_target['weather_type_tomorrow'].value_counts()
colors_wt = {'Sunny': 'gold', 'Cloudy': 'lightgray', 'Rainy': 'steelblue', 'Stormy': 'darkslategray'}
axes[3].bar(wt.index, wt.values,
            color=[colors_wt.get(k, 'gray') for k in wt.index])
axes[3].set_title('weather_type_tomorrow\n(4 classes)')
for bar, val in zip(axes[3].patches, wt.values):
    axes[3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=8)

plt.suptitle('Distribution des variables cibles', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions continues des cibles
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# max_temp_tomorrow
axes[0].hist(df_target['max_temp_tomorrow'].dropna(), bins=60, color='tomato', alpha=0.8, edgecolor='white')
axes[0].axvline(df_target['max_temp_tomorrow'].mean(), color='black', linestyle='--', linewidth=1.5,
                label=f"Moy {df_target['max_temp_tomorrow'].mean():.1f}°C")
axes[0].set_title('max_temp_tomorrow (°C)')
axes[0].legend()

# rain_tomorrow_proba
axes[1].hist(df_target['rain_tomorrow_proba'].dropna(), bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=1, label='Seuil 0.5')
axes[1].set_title('rain_tomorrow_proba')
axes[1].legend()

# comfort_score
axes[2].hist(df_target['comfort_score'].dropna(), bins=50, color='mediumseagreen', alpha=0.8, edgecolor='white')
axes[2].axvline(df_target['comfort_score'].median(), color='black', linestyle='--', linewidth=1.5,
                label=f"Médiane {df_target['comfort_score'].median():.0f}")
axes[2].set_title('comfort_score (0–100)')
axes[2].legend()

plt.suptitle('Distributions des cibles continues', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions des probabilités de risque
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

proba_cols = [
    ('rain_tomorrow_proba', 'steelblue', 'Probabilité pluie demain'),
    ('storm_probability', 'darkslategray', 'Probabilité tempête'),
    ('heatwave_risk', 'tomato', 'Score risque canicule'),
]

for ax, (col, color, title) in zip(axes, proba_cols):
    data = df_target[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Probabilité / Score')
    q90 = data.quantile(0.9)
    ax.axvline(q90, color='red', linestyle=':', linewidth=1.5,
               label=f'P90 = {q90:.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Distributions des probabilités de risque', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Saisonnalité & patterns temporels

In [ ]:
df['month'] = df['date'].dt.month
df['season'] = df['month'].map({
    12: 'Été', 1: 'Été', 2: 'Été',
    3: 'Automne', 4: 'Automne', 5: 'Automne',
    6: 'Hiver', 7: 'Hiver', 8: 'Hiver',
    9: 'Printemps', 10: 'Printemps', 11: 'Printemps',
})
MOIS_LABELS = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

# Températures par mois
monthly_temp = df.groupby('month')[['min_temp', 'max_temp']].mean()
monthly_temp.index = [MOIS_LABELS[i-1] for i in monthly_temp.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].fill_between(range(12), monthly_temp['min_temp'], monthly_temp['max_temp'],
                     alpha=0.3, color='tomato', label='Plage min-max')
axes[0].plot(range(12), monthly_temp['min_temp'], 'o-', color='steelblue', label='Temp min')
axes[0].plot(range(12), monthly_temp['max_temp'], 'o-', color='tomato', label='Temp max')
axes[0].set_xticks(range(12))
axes[0].set_xticklabels(MOIS_LABELS)
axes[0].set_title('Températures moyennes par mois (hémisphère sud)')
axes[0].set_ylabel('°C')
axes[0].legend()

# Probabilité de pluie par mois
monthly_rain = df.groupby('month')['rain_today'].mean() * 100
monthly_rain.index = [MOIS_LABELS[i-1] for i in monthly_rain.index]
axes[1].bar(range(12), monthly_rain.values, color='steelblue', alpha=0.8)
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(MOIS_LABELS)
axes[1].set_title('Probabilité de pluie par mois (%)')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

In [ ]:
# Évolution annuelle — température et précipitations
annual = df.groupby('year').agg(
    max_temp_mean=('max_temp', 'mean'),
    min_temp_mean=('min_temp', 'mean'),
    rain_days_pct=('rain_today', 'mean'),
    rainfall_mean=('rainfall', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(annual['year'], annual['max_temp_mean'], 'o-', color='tomato', label='Max temp')
axes[0].plot(annual['year'], annual['min_temp_mean'], 'o-', color='steelblue', label='Min temp')
# Tendance linéaire
for col, color in [('max_temp_mean', 'tomato'), ('min_temp_mean', 'steelblue')]:
    valid = annual[annual['year'] < 2026]
    slope, intercept, r, p, _ = stats.linregress(valid['year'], valid[col])
    trend = slope * valid['year'] + intercept
    axes[0].plot(valid['year'], trend, '--', color=color, alpha=0.5, linewidth=1.5,
                 label=f'Tendance {slope*10:.2f}°C/décennie')
axes[0].set_title('Évolution annuelle des températures')
axes[0].set_ylabel('°C')
axes[0].legend(fontsize=8)

axes[1].bar(annual['year'], annual['rain_days_pct'] * 100, color='steelblue', alpha=0.8)
axes[1].set_title('% jours de pluie par année')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots par saison
season_order = ['Été', 'Automne', 'Hiver', 'Printemps']
palette_seasons = {'Été': 'tomato', 'Automne': 'orange', 'Hiver': 'steelblue', 'Printemps': 'mediumseagreen'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='season', y='max_temp', order=season_order,
            palette=palette_seasons, ax=axes[0], flierprops={'markersize': 1})
axes[0].set_title('Max temp par saison')
axes[0].set_xlabel('')
axes[0].set_ylabel('°C')

sns.boxplot(data=df, x='season', y='rainfall', order=season_order,
            palette=palette_seasons, ax=axes[1], flierprops={'markersize': 1})
axes[1].set_title('Précipitations par saison')
axes[1].set_xlabel('')
axes[1].set_ylabel('mm')
axes[1].set_ylim(0, 30)

sns.boxplot(data=df, x='season', y='humidity_3pm', order=season_order,
            palette=palette_seasons, ax=axes[2], flierprops={'markersize': 1})
axes[2].set_title('Humidité 3pm par saison')
axes[2].set_xlabel('')
axes[2].set_ylabel('%')

plt.suptitle('Patterns saisonniers (hémisphère sud — été = déc-fév)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Géographie — patterns par ville et état

In [ ]:
# Statistiques par ville
city_stats = df.groupby(['city', 'state']).agg(
    max_temp_mean=('max_temp', 'mean'),
    min_temp_mean=('min_temp', 'mean'),
    rain_days_pct=('rain_today', 'mean'),
    rainfall_mean=('rainfall', 'mean'),
    humidity_3pm_mean=('humidity_3pm', 'mean'),
    wind_gust_mean=('wind_gust_speed', 'mean'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
).reset_index().sort_values('max_temp_mean', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

axes[0].barh(city_stats['city'], city_stats['max_temp_mean'], color='tomato', alpha=0.8)
axes[0].barh(city_stats['city'], city_stats['min_temp_mean'], color='steelblue', alpha=0.8)
axes[0].set_title('Températures moyennes par ville')
axes[0].set_xlabel('°C')
axes[0].legend(['Max temp', 'Min temp'])

city_rain_sorted = city_stats.sort_values('rain_days_pct', ascending=True)
axes[1].barh(city_rain_sorted['city'], city_rain_sorted['rain_days_pct'] * 100, color='steelblue', alpha=0.8)
axes[1].set_title('% jours de pluie par ville')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter géographique — latitude vs max_temp et précipitations
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sc1 = axes[0].scatter(
    city_stats['latitude'], city_stats['max_temp_mean'],
    c=city_stats['max_temp_mean'], cmap='Reds', s=80, alpha=0.9
)
for _, row in city_stats.iterrows():
    axes[0].annotate(row['city'], (row['latitude'], row['max_temp_mean']),
                     fontsize=6, ha='right')
axes[0].set_xlabel('Latitude (°S — négatif = sud)')
axes[0].set_ylabel('Max temp moyenne (°C)')
axes[0].set_title('Latitude vs Température max')
plt.colorbar(sc1, ax=axes[0], label='°C')

sc2 = axes[1].scatter(
    city_stats['latitude'], city_stats['rain_days_pct'] * 100,
    c=city_stats['rain_days_pct'], cmap='Blues', s=80, alpha=0.9
)
for _, row in city_stats.iterrows():
    axes[1].annotate(row['city'], (row['latitude'], row['rain_days_pct'] * 100),
                     fontsize=6, ha='right')
axes[1].set_xlabel('Latitude (°S)')
axes[1].set_ylabel('% jours de pluie')
axes[1].set_title('Latitude vs Fréquence de pluie')
plt.colorbar(sc2, ax=axes[1], label='% jours pluie')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap météo par état
state_stats = df.groupby('state').agg(
    max_temp=('max_temp', 'mean'),
    min_temp=('min_temp', 'mean'),
    rain_days=('rain_today', 'mean'),
    humidity_3pm=('humidity_3pm', 'mean'),
    wind_gust=('wind_gust_speed', 'mean'),
    sunshine=('sunshine_hours', 'mean'),
).round(2)

state_norm = (state_stats - state_stats.min()) / (state_stats.max() - state_stats.min())

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(state_norm.T, ax=ax, cmap='RdYlGn', annot=state_stats.T,
            fmt='.1f', linewidths=0.5)
ax.set_title('Profil météo par état (normalisé 0–1, valeurs = moyennes absolues)')
plt.tight_layout()
plt.show()

## 7. Corrélations entre features

In [ ]:
# Corrélations features × cibles numériques
numeric_features = [
    'min_temp', 'max_temp', 'temp_9am', 'temp_3pm',
    'rainfall', 'humidity_9am', 'humidity_3pm',
    'pressure_9am', 'pressure_3pm',
    'cloud_9am', 'cloud_3pm', 'sunshine_hours',
    'wind_gust_speed', 'wind_speed_9am', 'wind_speed_3pm',
    'rain_today',
    'rain_tomorrow', 'max_temp_tomorrow', 'heatwave_risk', 'frost_risk', 'storm_probability'
]

corr = df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, square=True)
ax.set_title('Matrice de corrélation features × cibles', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top corrélations avec rain_tomorrow
target_corr = corr['rain_tomorrow'].drop('rain_tomorrow').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if corr.loc[c, 'rain_tomorrow'] > 0 else 'steelblue' for c in target_corr.index]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_title('Corrélations absolues avec rain_tomorrow\n(rouge = corrélation positive)')
ax.set_xlabel('|Corrélation|')
ax.axvline(0.3, color='red', linestyle='--', linewidth=1, label='0.3')
ax.legend()
plt.tight_layout()
plt.show()

print("Top 5 corrélations positives avec rain_tomorrow :")
top5_pos = corr['rain_tomorrow'].drop('rain_tomorrow').sort_values(ascending=False).head(5)
print(top5_pos.to_string())
print("\nTop 5 corrélations négatives avec rain_tomorrow :")
top5_neg = corr['rain_tomorrow'].drop('rain_tomorrow').sort_values().head(5)
print(top5_neg.to_string())

In [ ]:
# Pairplot des 4 features les plus corrélées avec rain_tomorrow
top_features = target_corr.head(4).index.tolist() + ['rain_tomorrow']
sample = df[top_features].dropna().sample(min(3000, len(df)), random_state=42)

fig = sns.pairplot(
    sample,
    hue='rain_tomorrow',
    palette={0.0: 'steelblue', 1.0: 'tomato'},
    diag_kind='kde',
    plot_kws={'alpha': 0.3, 's': 10},
    diag_kws={'fill': True}
)
fig.figure.suptitle('Pairplot — features les plus corrélées avec rain_tomorrow', y=1.02, fontweight='bold')
plt.show()

## 8. Analyse des prédictions du modèle

In [ ]:
# Calibration : probabilité prédite vs fréquence réelle (courbe de calibration)
df_cal = df[df['rain_tomorrow'].notna() & df['rain_tomorrow_proba'].notna()].copy()
df_cal['proba_bin'] = pd.cut(df_cal['rain_tomorrow_proba'], bins=10)
cal_curve = df_cal.groupby('proba_bin', observed=False).agg(
    freq_reelle=('rain_tomorrow', 'mean'),
    n=('rain_tomorrow', 'count')
).reset_index()
cal_curve['bin_center'] = cal_curve['proba_bin'].apply(lambda x: x.mid)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Calibration parfaite')
ax.scatter(cal_curve['bin_center'], cal_curve['freq_reelle'],
           s=cal_curve['n'] / cal_curve['n'].max() * 300 + 20,
           color='tomato', alpha=0.8, label='Calibration modèle (taille ∝ n)')
ax.plot(cal_curve['bin_center'], cal_curve['freq_reelle'], 'o-', color='tomato', linewidth=1.5)
ax.set_xlabel('Probabilité prédite')
ax.set_ylabel('Fréquence réelle de pluie')
ax.set_title('Courbe de calibration — rain_tomorrow_proba')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des prédictions par type météo (weather_type_tomorrow)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

wt_order = ['Sunny', 'Cloudy', 'Rainy', 'Stormy']
wt_palette = {'Sunny': 'gold', 'Cloudy': 'lightgray', 'Rainy': 'steelblue', 'Stormy': 'darkslategray'}

sns.boxplot(data=df_cal, x='weather_type_tomorrow', y='rain_tomorrow_proba',
            order=wt_order, palette=wt_palette, ax=axes[0],
            flierprops={'markersize': 1})
axes[0].set_title('Proba pluie par type météo prédit')
axes[0].set_xlabel('')

sns.boxplot(data=df_cal, x='weather_type_tomorrow', y='max_temp_tomorrow',
            order=wt_order, palette=wt_palette, ax=axes[1],
            flierprops={'markersize': 1})
axes[1].set_title('Max temp demain par type météo')
axes[1].set_xlabel('')
axes[1].set_ylabel('°C')

sns.boxplot(data=df_cal, x='weather_type_tomorrow', y='comfort_score',
            order=wt_order, palette=wt_palette, ax=axes[2],
            flierprops={'markersize': 1})
axes[2].set_title('Comfort score par type météo')
axes[2].set_xlabel('')

plt.suptitle('Cohérence des prédictions entre les sorties du modèle', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Prédictions de risque par ville (moyennes)
risk_by_city = df_cal.groupby('city').agg(
    heatwave_risk_mean=('heatwave_risk', 'mean'),
    frost_risk_mean=('frost_risk', 'mean'),
    storm_prob_mean=('storm_probability', 'mean'),
).sort_values('heatwave_risk_mean', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 9))

axes[0].barh(risk_by_city.index, risk_by_city['heatwave_risk_mean'], color='tomato', alpha=0.8)
axes[0].set_title('Risque canicule moyen par ville')
axes[0].set_xlabel('Score moyen')

risk_frost = risk_by_city.sort_values('frost_risk_mean', ascending=True)
axes[1].barh(risk_frost.index, risk_frost['frost_risk_mean'], color='lightskyblue', alpha=0.9)
axes[1].set_title('Risque gel moyen par ville')
axes[1].set_xlabel('Score moyen')

risk_storm = risk_by_city.sort_values('storm_prob_mean', ascending=True)
axes[2].barh(risk_storm.index, risk_storm['storm_prob_mean'], color='darkslategray', alpha=0.8)
axes[2].set_title('Probabilité tempête moyenne par ville')
axes[2].set_xlabel('Score moyen')

plt.suptitle('Profil de risque par ville', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cohérence : max_temp_tomorrow vs max_temp actuel (scatter)
sample_sc = df_cal[['max_temp', 'max_temp_tomorrow', 'state']].dropna().sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    sample_sc['max_temp'], sample_sc['max_temp_tomorrow'],
    c=sample_sc['max_temp_tomorrow'] - sample_sc['max_temp'],
    cmap='RdBu_r', alpha=0.4, s=8, vmin=-10, vmax=10
)
ax.plot([5, 50], [5, 50], 'k--', linewidth=1, label='Identité (pas de changement)')
plt.colorbar(scatter, ax=ax, label='Δ temp (°C)')
ax.set_xlabel('Max temp aujourd\'hui (°C)')
ax.set_ylabel('Max temp prédit demain (°C)')
ax.set_title('Cohérence : temp aujourd\'hui vs prédiction demain')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

delta = df_cal['max_temp_tomorrow'] - df_cal['max_temp']
print(f"Δ temp (demain − auj.) : moy={delta.mean():.2f}°C  std={delta.std():.2f}°C  p5={delta.quantile(0.05):.1f}°C  p95={delta.quantile(0.95):.1f}°C")

## 9. Synthèse

In [ ]:
print("=" * 65)
print("SYNTHÈSE EXPLORATOIRE — weather_final.csv")
print("=" * 65)

print(f"""
DATASET
  Lignes           : {len(df):,}
  Villes           : {df['city'].nunique()} villes / {df['state'].nunique()} états
  Période          : {df['date'].min().date()} → {df['date'].max().date()}
  Colonnes Open-Meteo enrichies : ~{len(df_recent):,} lignes ({len(df_recent)/len(df):.1%})

QUALITÉ
  Colonnes base    : 0 NaN (min_temp, max_temp, pressure, humidity, rainfall...)
  Colonnes enrichies : ~80% NaN (données historiques non disponibles)
  Targets NaN      : 26 lignes manquantes (derniers jours sans label J+1)

CLASSE IMBALANCE (targets binaires)
  rain_tomorrow    : {df_target['rain_tomorrow'].mean():.1%} positifs
  heatwave_risk    : {(df_target['heatwave_risk'].round() == 1).mean():.1%} à risque
  frost_risk       : {(df_target['frost_risk'].round() == 1).mean():.1%} à risque
  weather_type     : Sunny={df_target['weather_type_tomorrow'].eq('Sunny').mean():.1%}  Cloudy={df_target['weather_type_tomorrow'].eq('Cloudy').mean():.1%}  Rainy={df_target['weather_type_tomorrow'].eq('Rainy').mean():.1%}  Stormy={df_target['weather_type_tomorrow'].eq('Stormy').mean():.1%}

FEATURES LES PLUS CORRÉLÉES À rain_tomorrow
  {corr['rain_tomorrow'].drop('rain_tomorrow').abs().sort_values(ascending=False).head(3).to_string()}

SAISONNALITÉ
  Max temp : été austral (déc-fév) ~26°C, hiver (juin-août) ~17°C
  Pluie    : fréquence plus élevée en hiver et automne pour la plupart des villes

PRÉDICTIONS
  Calibration rain_proba  : bonne (courbe proche de la diagonale)
  Cohérence weather_type  : distributions cohérentes avec proba pluie et temp
  Risque canicule         : max dans les villes du nord (QLD, NT)
  Risque gel              : max dans les villes du sud (VIC, TAS, NSW montagne)
""")

print("=" * 65)